<a href="https://colab.research.google.com/github/yy726/tool_use_demo/blob/main/notebooks/03_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Demo: try the planner

Serves the trained checkpoints behind OpenAI-compatible endpoints and opens a small web app where reviewers can run the
five preset hard queries, or their own, and compare checkpoints side by side.

How the pieces fit:

- **Model server.** `SERVER = "unsloth"` runs Unsloth Studio's `unsloth run` on the GGUF builds pushed by the training
  notebooks. `SERVER = "vllm"` serves the merged weights with vLLM instead. `SERVER = "external"` uses endpoints you
  already run somewhere (for example Unsloth Desktop on your own machine).
- **Tool loop.** Unsloth's chat window can only run its own built-in tools, not our four. So this notebook's client
  sends our tool schemas with each request, executes the calls the model asks for against the local mock world and
  knowledge base, and returns the results - at most three rounds.

A T4 is enough for `SERVER = "unsloth"`; `SERVER = "vllm"` needs an L4 or A100.

In [ ]:
HF_USER = ""                 # your Hugging Face user name ("" = take it from the HF_TOKEN secret)
MODEL_KEY = "qwen3-1.7b"     # must match the training notebooks
SMOKE = None                 # None: serve the full-run repositories if they exist, else the "-smoke" ones. True / False: force one.
STAGES = ["sft", "grpo"]     # trained checkpoints to serve
INCLUDE_BASE = True          # also serve the untuned base model for comparison
SERVER = "unsloth"           # "unsloth" | "vllm" | "external"
EXTERNAL = {}                # for SERVER = "external": {"grpo": ("https://host/v1", "sk-unsloth-...")}
GGUF_VARIANT = "Q8_0"
SHARE = True                 # public Gradio link for reviewers (valid while this runtime is alive)

## 1. Install and set up

In [14]:
!pip install -q openai gradio bm25s==0.3.16 "datasets>=4.7.0,<5.0.0" huggingface_hub
if SERVER == "unsloth":
    !curl -fsSL https://unsloth.ai/install.sh | sh
elif SERVER == "vllm":
    !pip install -q vllm


  🦥 Unsloth Studio Installer
  ────────────────────────────────────────────────────

  platform       linux
  deps           all system dependencies found
  sandbox        bubblewrap works: tool calls run in an OS sandbox
  uv cache       reusing this install's Unsloth Studio cache (/root/.unsloth/studio/cache/uv)
                 found unsloth 2026.11.4
                 preserving existing environment for rollback...
                 previous environment preserved for rollback
  venv           creating Python 3.13 virtual environment
                 /root/.unsloth/studio/unsloth_studio
  venv           using environment
                 /root/.unsloth/studio/unsloth_studio
                 existing install has torch 2.13.0+cu130 -- keeping it (set UNSLOTH_TORCH_UPGRADE=1 to get the newest release)
  gpu            NVIDIA A100-SXM4-80GB (sm_80)
                 Driver: 580.82.07
                 installing PyTorch (https://download.pytorch.org/whl/cu130)...
                 installin

In [15]:
import os, subprocess, sys

REPO_URL = "https://github.com/yy726/tool_use_demo.git"
REPO_BRANCH = "main"

REPO_DIR = "/content/tool_use_demo"
if os.getcwd() != REPO_DIR and os.path.isdir("planner"):        # run from your own checkout: use it as it is
    REPO_DIR = os.getcwd()
elif os.getcwd() != REPO_DIR and os.path.isdir("../planner"):
    REPO_DIR = os.path.abspath("..")
elif not os.path.isdir(REPO_DIR):
    url = REPO_URL
    try:                                          # private repo: add a GITHUB_TOKEN secret
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token: url = url.replace("https://", f"https://{token}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, REPO_DIR], check=True)
else:                                             # a clone from an earlier run: move it to the latest commit of REPO_BRANCH
    for cmd in (["fetch", "--depth", "1", "origin", REPO_BRANCH], ["checkout", "-f", "-B", REPO_BRANCH, "FETCH_HEAD"]):
        done = subprocess.run(["git", "-C", REPO_DIR] + cmd, capture_output=True, text=True)
        if done.returncode: print("WARNING: git", cmd[0], "failed, the code in", REPO_DIR, "may be stale:", done.stderr.strip())
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
for _name in [m for m in sys.modules if m == "planner" or m.startswith("planner.")]:
    del sys.modules[_name]                        # a session that already imported the old code picks up the new code
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "@", head)

repo: /content/tool_use_demo @ f678a44 Notebooks: refresh the Colab clone on every run


In [16]:
from planner.config import get_config
from planner.persist import read_secrets
from planner.setup_env import setup_environment

MODES = ["a100", "smoke"] if SMOKE is None else ["smoke" if SMOKE else "a100"]
cfg = get_config("a100", MODEL_KEY)
read_secrets(("HF_TOKEN",))
if not HF_USER and os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    HF_USER = whoami(token=os.environ["HF_TOKEN"])["name"]
assert HF_USER or SERVER == "external", "Set HF_USER (or add an HF_TOKEN secret)."

# The world and the full knowledge base the preset queries were picked from (about a minute to build).
world, corpus = setup_environment(cache_dir="/content/planner_env", seed=cfg.seed)

Building the knowledge base from HotpotQA (8000 train + 1500 validation questions) ...
Knowledge base: 83438 passages. World: 90 regions, 270 cities.


## 2. Start the model servers

Each checkpoint gets its own port. With `unsloth run`, the API key it prints on start-up is read from its log.

In [17]:
import shutil
from planner.demo import OpenAIBackend
from planner.serve import Endpoint, first_existing_repo, start_unsloth, start_vllm, wait_ready

def hub_repo(stage, kind=""):
    """The pushed repository of a checkpoint: from the full run, else from the SMOKE run."""
    return first_existing_repo([get_config(mode, MODEL_KEY).repo(HF_USER, stage, kind) for mode in MODES])

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.path.expanduser("~/.unsloth/bin") + os.pathsep + os.environ["PATH"]
names = (["base"] if INCLUDE_BASE else []) + STAGES
endpoints = {}
if SERVER == "external":
    endpoints = {name: Endpoint(name, url, key) for name, (url, key) in EXTERNAL.items()}
else:
    for i, name in enumerate(names):
        port, log = 8001 + i, f"/content/server_{name}.log"
        try:
            if SERVER == "unsloth":
                assert shutil.which("unsloth"), "the `unsloth` command is not on PATH: check the install cell's output"
                repo = f"unsloth/{cfg.base_model_hf.split('/')[-1]}-GGUF" if name == "base" else hub_repo(name, "GGUF")
                endpoints[name] = wait_ready(start_unsloth(name, repo, port, log, GGUF_VARIANT), key_from_log=True)
            else:
                repo = cfg.base_model_hf if name == "base" else hub_repo(name)
                endpoints[name] = wait_ready(start_vllm(name, repo, port, log, gpu_memory_utilization=round(0.85 / len(names), 2)))
            print(f"{name}: {repo} ready at {endpoints[name].base_url}")
        except Exception as error:
            print(f"{name}: could not start ({error})")

backends = {name: OpenAIBackend(e.base_url, e.api_key) for name, e in endpoints.items()}
assert backends, "No endpoint is up. Check /content/server_*.log, or try SERVER = 'vllm'."
print({name: b.model for name, b in backends.items()})

$ unsloth run --model unsloth/Qwen3-1.7B-GGUF:Q8_0 -p 8001 -y --disable-tools > /content/server_base.log
base: unsloth/Qwen3-1.7B-GGUF ready at http://127.0.0.1:8001/v1
$ unsloth run --model darcyyao/qwen3-1.7b-planner-smoke-sft-GGUF:Q8_0 -p 8002 -y --disable-tools > /content/server_sft.log
sft: darcyyao/qwen3-1.7b-planner-smoke-sft-GGUF ready at http://127.0.0.1:8002/v1
$ unsloth run --model darcyyao/qwen3-1.7b-planner-smoke-grpo-GGUF:Q8_0 -p 8003 -y --disable-tools > /content/server_grpo.log
grpo: darcyyao/qwen3-1.7b-planner-smoke-grpo-GGUF ready at http://127.0.0.1:8003/v1
{'base': 'unsloth/Qwen3-1.7B-GGUF', 'sft': 'darcyyao/qwen3-1.7b-planner-smoke-sft-GGUF', 'grpo': 'darcyyao/qwen3-1.7b-planner-smoke-grpo-GGUF'}


## 3. Five hard queries

| # | type | what a good plan does |
|---|---|---|
| 1 | bridge / multi-hop | search, then search again with an entity found in the first result |
| 2 | decomposition + fan-out + synthesis | look up two entities in parallel, then compare |
| 3 | multi-date fan-out + calculation | three weather lookups in one round, then the calculator |
| 4 | cross-tool bridge + tool selection | `get_election` gives the capital and the date for `get_weather` |
| 5 | multi-part ask + query reformulation | split the question; the knowledge part needs a follow-up query |

They come from the held-out set, so each has a reference plan and gold answer, and the output shows the relevance and
factuality scores along with failure tags.

In [18]:
from IPython.display import Markdown, display
from planner.demo import answer, format_expected, format_markdown, load_examples

examples = load_examples()
for i, example in enumerate(examples, 1):
    task = example["task"]
    display(Markdown(f"## {i}. {example['label']}\n\n> {task['query']}\n\n{example['why']}\n\n{format_expected(task)}"))
    for name, backend in backends.items():
        try:
            display(Markdown(format_markdown(answer(task, backend), title=name)))
        except Exception as error:
            print(f"{name}: request failed ({error})")

## 1. Bridge / multi-hop

> What year was the author of Cock and Bull born?

The second search depends on an entity found by the first one.

**Reference plan:** [knowledge_search({"query": "Cock and Bull (book)"})] -> [knowledge_search({"query": "Will Self"})]

**Reference answer:** 1961

DEBUG:bm25s:Building index from IDs objects


### base

**Round 1** (1 call)

- `knowledge_search({"query": "author of Cock and Bull"})`

  - [{"doc_id": "D078285", "title": "Cock and Bull (book)", "text": "Cock and Bull is the title of a volume composed of two novellas by Will Self, which includes the stories \"Cock\" and \"Bull\". The two stories are characterized by empty, emotionless, phatic sex; rape; cruelty; and violence. The book was originally published in 1992 by Bloomsbury."}, {"doc_id": "D061234", "title": "A Cock and Bull Story", "text": "A Cock and Bull Story (marketed in Australia, New Zealand and the United States as Tristram Shandy: A Cock and Bull Story) is a 2005 British comedy film directed by Michael Winterbotto ...

**Final:** Answer: 1927

_Tool cost: 1.00 in 1 round(s)_

**Score:** bad - relevance 4/4, factuality 1/4, tags: multi_hop_incomplete

### sft

**Round 1** (1 call)

- `knowledge_search({"query": "Cock and Bull author"})`

  - [{"doc_id": "D078285", "title": "Cock and Bull (book)", "text": "Cock and Bull is the title of a volume composed of two novellas by Will Self, which includes the stories \"Cock\" and \"Bull\". The two stories are characterized by empty, emotionless, phatic sex; rape; cruelty; and violence. The book was originally published in 1992 by Bloomsbury."}, {"doc_id": "D061234", "title": "A Cock and Bull Story", "text": "A Cock and Bull Story (marketed in Australia, New Zealand and the United States as Tristram Shandy: A Cock and Bull Story) is a 2005 British comedy film directed by Michael Winterbotto ...

**Final:** Answer: 1927

_Tool cost: 1.00 in 1 round(s)_

**Score:** bad - relevance 4/4, factuality 1/4, tags: multi_hop_incomplete

### grpo

**Round 1** (1 call)

- `knowledge_search({"query": "Cock and Bull author"})`

  - [{"doc_id": "D078285", "title": "Cock and Bull (book)", "text": "Cock and Bull is the title of a volume composed of two novellas by Will Self, which includes the stories \"Cock\" and \"Bull\". The two stories are characterized by empty, emotionless, phatic sex; rape; cruelty; and violence. The book was originally published in 1992 by Bloomsbury."}, {"doc_id": "D061234", "title": "A Cock and Bull Story", "text": "A Cock and Bull Story (marketed in Australia, New Zealand and the United States as Tristram Shandy: A Cock and Bull Story) is a 2005 British comedy film directed by Michael Winterbotto ...

**Final:** Answer: 1927

_Tool cost: 1.00 in 1 round(s)_

**Score:** bad - relevance 4/4, factuality 1/4, tags: multi_hop_incomplete

## 2. Decomposition + fan-out + synthesis

> Which genus has more species, Bactris and Epigaea?

Two entities must be looked up in parallel and then compared.

**Reference plan:** [knowledge_search({"query": "Bactris"}), knowledge_search({"query": "Epigaea"})]

**Reference answer:** Bactris

... (11 outputs left out)

## 5. Multi-part ask: fan-out + query reformulation

> Two questions. First: Who was the runner-up in Fenonia's 2024 election? Second: What kind of comedian was Michael Mclntyre's father?

Searching with the whole question retrieves nothing useful; it has to be split, and the knowledge part needs a follow-up query.

**Reference plan:** [knowledge_search({"query": "Ray Cameron (comedian)"}), get_election({"region": "Fenonia", "year": 2024})] -> [knowledge_search({"query": "Michael McIntyre"})]

**Reference answer:** Canadian comedian; Greta Lindqvist

### base

**Round 1** (2 calls, parallel)

- `get_election({"region": "Fenonia", "year": 2024})`

  - {"region": "Fenonia", "year": 2024, "winner": "Zelda Stroud", "winner_party": "Liberty Coalition", "winner_vote_share_pct": 51.3, "runner_up": "Greta Lindqvist", "runner_up_party": "Unity Party", "runner_up_vote_share_pct": 42.0, "turnout_pct": 63.5, "election_date": "2024-04-20", "capital": "Harhaven"}

- `knowledge_search({"query": "Michael Mclntyre's father"})`

  - [{"doc_id": "D080076", "title": "Michael Redmond (comedian)", "text": "Michael Redmond (born October 1950) is an Irish stand-up comedian from Blackrock, Dublin, known for playing Father Stone in the \"Father Ted\" episode \"Entertaining Father Stone\". He has a deadpan style of delivery and has been described as the \"possessor of comedy's most mournful moustache\"."}, {"doc_id": "D013302", "title": "Michael Hawkins (American actor)", "text": "Michael Hawkins (also known as Michael D. Gainsborough; born Thomas Knight Slater on December 26, 1938) is an American actor. He is the father of actor  ...

**Final:** Answer: The runner-up in Fenonia's 2024 election was Greta Lindqvist. Michael Mclntyre's father is not explicitly mentioned in the provided knowledge base, but the closest match is Michael Hawkins, who is noted as the father of actor Christian Slater.

_Tool cost: 1.25 in 1 round(s)_

**Score:** bad - relevance 2/4, factuality 2/4, tags: not_decomposed, query_rewrite_failed

### sft

**Round 1** (1 call)

- `get_election({"region": "Fenonia", "year": 2024})`

  - {"region": "Fenonia", "year": 2024, "winner": "Zelda Stroud", "winner_party": "Liberty Coalition", "winner_vote_share_pct": 51.3, "runner_up": "Greta Lindqvist", "runner_up_party": "Unity Party", "runner_up_vote_share_pct": 42.0, "turnout_pct": 63.5, "election_date": "2024-04-20", "capital": "Harhaven"}

**Round 2** (1 call)

- `knowledge_search({"query": "Michael Mclntyre's father"})`

  - [{"doc_id": "D080076", "title": "Michael Redmond (comedian)", "text": "Michael Redmond (born October 1950) is an Irish stand-up comedian from Blackrock, Dublin, known for playing Father Stone in the \"Father Ted\" episode \"Entertaining Father Stone\". He has a deadpan style of delivery and has been described as the \"possessor of comedy's most mournful moustache\"."}, {"doc_id": "D013302", "title": "Michael Hawkins (American actor)", "text": "Michael Hawkins (also known as Michael D. Gainsborough; born Thomas Knight Slater on December 26, 1938) is an American actor. He is the father of actor  ...

**Final:** Answer: Michael Redmond

_Tool cost: 1.25 in 2 round(s)_

**Score:** bad - relevance 2/4, factuality 1/4, tags: not_decomposed, query_rewrite_failed

### grpo

**Round 1** (1 call)

- `get_election({"region": "Fenonia", "year": 2024})`

  - {"region": "Fenonia", "year": 2024, "winner": "Zelda Stroud", "winner_party": "Liberty Coalition", "winner_vote_share_pct": 51.3, "runner_up": "Greta Lindqvist", "runner_up_party": "Unity Party", "runner_up_vote_share_pct": 42.0, "turnout_pct": 63.5, "election_date": "2024-04-20", "capital": "Harhaven"}

**Round 2** (1 call)

- `knowledge_search({"query": "Michael Mclntyre's father"})`

  - [{"doc_id": "D080076", "title": "Michael Redmond (comedian)", "text": "Michael Redmond (born October 1950) is an Irish stand-up comedian from Blackrock, Dublin, known for playing Father Stone in the \"Father Ted\" episode \"Entertaining Father Stone\". He has a deadpan style of delivery and has been described as the \"possessor of comedy's most mournful moustache\"."}, {"doc_id": "D013302", "title": "Michael Hawkins (American actor)", "text": "Michael Hawkins (also known as Michael D. Gainsborough; born Thomas Knight Slater on December 26, 1938) is an American actor. He is the father of actor  ...

**Final:** Answer: Michael Redmond

_Tool cost: 1.25 in 2 round(s)_

**Score:** bad - relevance 2/4, factuality 1/4, tags: not_decomposed, query_rewrite_failed

## 4. Web app for reviewers

Pick a preset or type any question. With `SHARE = True` Gradio prints a public `gradio.live` link that works while this
runtime is running. Weather and elections belong to a fictional world (regions such as the ones in the presets), so
ask about those; `knowledge_search` covers the Wikipedia paragraphs of the HotpotQA questions used to build the index.

In [19]:
from planner.demo import build_ui

ui = build_ui(backends, examples)
ui.launch(share=SHARE, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a9121686649080d1c2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Hosting it yourself with Unsloth Desktop / Studio

On your own machine, with [Unsloth](https://unsloth.ai/docs/get-started/unsloth-studio) installed:

```bash
unsloth run --model <hf-user>/qwen3-1.7b-planner-grpo-GGUF:Q8_0 -p 8888 --disable-tools
```

It prints the endpoint URL and an API key. Then, from a clone of this repository:

```bash
python -m planner.demo --base-url http://127.0.0.1:8888/v1 --api-key sk-unsloth-... --example 4
```

`unsloth studio --secure` publishes the endpoint on a Cloudflare HTTPS URL for remote reviewers; use that URL as
`--base-url` (the client already sends `stream: false`, which the tunnel requires). You can also load the GGUF in the
Unsloth chat window to talk to the model directly, but its tool calls will not be executed there: only the client in
this repository runs the four planner tools.

When you are done, stop the servers:

In [ ]:
for endpoint in endpoints.values(): endpoint.stop()